# 📘 AI Terminology — Simple Definition Teller (Google Colab Notebook)

This notebook builds, trains, evaluates, and tests the **AI Terminology Simple Definition Teller** model end-to-end in **Google Colab**:
1. **Downloads Real External Datasets:**
   - **Hugging Face `J0nasW/AI-Dictionary`** (`16,275` real AI keyphrases and weights).
   - **Official Google Machine Learning Glossary** (`733+` authoritative, beginner-friendly AI/ML terms and definitions).
2. **Builds & Explores `ai_dictionary_dataset.csv`** (cross-linking official definitions with noisy keyword variants like `back propagation`, `graph neuralnetworks`, etc.).
3. **Trains the Hybrid Neural MLP (`MLPClassifier`) + Character/Word N-Gram TF-IDF Model** (`train.py` logic).
4. **Evaluates Top-1 & Top-3 Accuracy** on student typos, spacing variations, and abbreviations.
5. **Interactive Simple Definition Teller** right inside Colab + exports `model.pkl` and `ai_dictionary_dataset.csv` for your GitHub & Streamlit app.

## 📦 Step 1: Import Libraries

In [ ]:
import os
import re
import json
import html
import pickle
import urllib.request
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neural_network import MLPClassifier
from sklearn.metrics.pairwise import cosine_similarity

print("All libraries imported successfully!")

## 🌐 Step 2: Download `J0nasW/AI-Dictionary` (Hugging Face) & Google ML Glossary

In [ ]:
J0NASW_CSV_PATH = "j0nasw_ai_dictionary.csv"
FINAL_CSV_PATH = "ai_dictionary_dataset.csv"
MODEL_PATH = "model.pkl"

def download_j0nasw_ai_dictionary():
    """Downloads J0nasW/AI-Dictionary (16,275 rows) from Hugging Face Hub."""
    url = "https://huggingface.co/datasets/J0nasW/AI-Dictionary/resolve/main/dictionary_weights_w_embeddings.csv"
    raw_file = "dictionary_weights_w_embeddings.csv"
    print(f"Downloading J0nasW/AI-Dictionary from Hugging Face ({url})...")
    urllib.request.urlretrieve(url, raw_file)
    df_j = pd.read_csv(raw_file, usecols=["keyword", "weight"])
    df_j.to_csv(J0NASW_CSV_PATH, index=False, encoding="utf-8")
    if os.path.exists(raw_file):
        os.remove(raw_file)
    print(f"Saved {len(df_j):,} AI keyphrases to {J0NASW_CSV_PATH}")
    return df_j

df_j0nasw = download_j0nasw_ai_dictionary()
df_j0nasw.head(10)

In [ ]:
def clean_html_text(raw_html: str) -> str:
    text = re.sub(r"<script.*?>.*?</script>", " ", raw_html, flags=re.S | re.I)
    text = re.sub(r"<style.*?>.*?</style>", " ", raw_html, flags=re.S | re.I)
    text = re.sub(r"<[^>]+>", " ", text)
    text = html.unescape(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def fetch_google_ml_glossary() -> list[dict]:
    url = "https://developers.google.com/machine-learning/glossary"
    print(f"Downloading Official Google Machine Learning Glossary from {url} ...")
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=30) as resp:
        page_html = resp.read().decode("utf-8", errors="ignore")

    pattern = re.compile(
        r'<h2[^>]*class="[^"]*hide-from-toc[^"]*"[^>]*>(.*?)</h2>(.*?)(?=<h2[^>]*class="[^"]*hide-from-toc|<h1|$)',
        re.S | re.I,
    )
    entries = []
    for match in pattern.finditer(page_html):
        raw_term = clean_html_text(match.group(1))
        raw_body = match.group(2)
        paragraphs = re.findall(r"<p[^>]*>(.*?)</p>", raw_body, re.S | re.I)
        cleaned_paras = []
        for p in paragraphs:
            cp = clean_html_text(p)
            if cp and not cp.lower().startswith(("see the ", "contrast with ", "for more information")):
                cleaned_paras.append(cp)

        if not raw_term or not cleaned_paras:
            continue

        entries.append({
            "term": raw_term,
            "simple_definition": cleaned_paras[0],
            "extra_context": cleaned_paras[1] if len(cleaned_paras) > 1 else "",
            "source": "Google Machine Learning Glossary + J0nasW/AI-Dictionary",
        })

    print(f"Extracted {len(entries)} official AI/ML terms & definitions!")
    return entries

def infer_category(term: str, definition: str) -> str:
    text = f"{term} {definition}".lower()
    if any(k in text for k in ["transformer", "token", "language model", "llm", "nlp", "word", "text", "bert", "gpt", "translation", "prompt"]):
        return "Natural Language Processing & LLMs"
    if any(k in text for k in ["image", "pixel", "convolutional", "cnn", "bounding box", "vision", "object detection"]):
        return "Computer Vision"
    if any(k in text for k in ["reinforcement", "reward", "agent", "policy", "markov", "q-learning", "action"]):
        return "Reinforcement Learning"
    if any(k in text for k in ["neural network", "layer", "neuron", "backpropagation", "activation", "relu", "sigmoid", "hidden layer", "deep"]):
        return "Deep Learning & Neural Networks"
    if any(k in text for k in ["gradient", "loss", "optimizer", "learning rate", "converge", "regularization", "epoch", "batch"]):
        return "Optimization & Training"
    if any(k in text for k in ["precision", "recall", "accuracy", "roc", "auc", "confusion matrix", "f1", "error", "metric", "validation"]):
        return "Evaluation & Metrics"
    if any(k in text for k in ["feature", "dataset", "missing", "outlier", "normalization", "scaling", "imbalance", "sampling"]):
        return "Data Science & Feature Engineering"
    if any(k in text for k in ["fairness", "bias", "ethics", "safety", "explainab", "privacy", "hallucination"]):
        return "AI Ethics, Safety & Explainability"
    return "Core Machine Learning"

# Build combined dataset and link with J0nasW/AI-Dictionary keywords
google_entries = fetch_google_ml_glossary()

compact_to_variants = {}
for kw in df_j0nasw["keyword"].astype(str):
    comp = re.sub(r"[^a-z0-9]", "", kw.lower())
    if comp:
        compact_to_variants.setdefault(comp, []).append(kw)

# Essential student aliases (e.g. back propagation -> backpropagation)
student_alias_boost = {
    "backpropagation": "back propagation, backprop, backward propagation",
    "stochastic gradient descent": "sgd, mini-batch gradient descent",
    "convolutional neural network": "cnn, convnet, conv net",
    "long short-term memory": "lstm, lstm network",
    "gated recurrent unit": "gru, gru network",
    "retrieval-augmented generation": "rag, retrieval augmented generation",
    "reinforcement learning from human feedback": "rlhf",
    "low-rank adaptation": "lora, qlora",
    "gradient boosting": "xgboost, lightgbm, gbdt",
}

rows = []
seen = set()
for item in google_entries:
    term = item["term"]
    clean_t = re.sub(r"\s*\([^)]*\)", "", term).strip().lower()
    if clean_t in seen or len(clean_t) < 2:
        continue
    seen.add(clean_t)

    comp_t = re.sub(r"[^a-z0-9]", "", clean_t)
    acronyms = [a.lower() for a in re.findall(r"\(([^)]+)\)", term)]
    aliases_set = {clean_t, *acronyms}

    for k_sub, extra_a in student_alias_boost.items():
        if k_sub in clean_t:
            for a in extra_a.split(","):
                aliases_set.add(a.strip())

    for prefix in ["back", "over", "under", "hyper", "multi", "auto", "down", "up", "cross", "self", "fine", "zero", "few"]:
        if clean_t.startswith(prefix) and len(clean_t) > len(prefix) + 3:
            aliases_set.add(f"{prefix} {clean_t[len(prefix):]}")

    for m in compact_to_variants.get(comp_t, [])[:5]:
        aliases_set.add(m)

    cat = infer_category(term, item["simple_definition"])
    rows.append({
        "term": term,
        "category": cat,
        "aliases": ", ".join(sorted(aliases_set)),
        "simple_definition": item["simple_definition"],
        "analogy": item["extra_context"] if item["extra_context"] else f"Core concept in {cat}.",
        "source": item["source"],
    })

df = pd.DataFrame(rows).sort_values("term").reset_index(drop=True)
df.to_csv(FINAL_CSV_PATH, index=False, encoding="utf-8")
print(f"Saved {len(df)} terms to {FINAL_CSV_PATH}")
df.head(10)

## 📊 Step 3: Exploratory Data Analysis (Category Distribution)

In [ ]:
plt.figure(figsize=(10, 5))
cat_counts = df["category"].value_counts()
cat_counts.sort_values().plot(kind="barh", color="#38bdf8", edgecolor="black")
plt.title("Distribution of AI/ML Terminology Across Domains", fontsize=14, fontweight="bold")
plt.xlabel("Number of Terms")
plt.ylabel("AI Domain Category")
plt.grid(axis="x", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

## 🧠 Step 4: Train Hybrid Neural MLP + Character/Word N-Gram TF-IDF Model

In [ ]:
def normalize_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def compact_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    return re.sub(r"[^a-z0-9]", "", text.lower())

def generate_typo_variants(phrase: str, rng: np.random.Generator) -> list[str]:
    norm = normalize_text(phrase)
    compact = compact_text(phrase)
    variants = {norm, compact}

    if " " in norm:
        variants.add(norm.replace(" ", ""))
    elif len(norm) >= 8:
        mid = len(norm) // 2
        variants.add(norm[:mid] + " " + norm[mid:])

    if len(norm) >= 5:
        idxs = rng.choice(len(norm), size=min(2, len(norm)), replace=False)
        for idx in idxs:
            if norm[idx] != " ":
                variants.add(norm[:idx] + norm[idx + 1 :])
            if idx < len(norm) - 1 and norm[idx] != " " and norm[idx + 1] != " ":
                swapped = norm[:idx] + norm[idx + 1] + norm[idx] + norm[idx + 2 :]
                variants.add(swapped)

    return [v for v in variants if len(v) >= 2]

def build_training_corpus(df: pd.DataFrame):
    rng = np.random.default_rng(42)
    X_texts, y_labels = [], []
    for idx, row in df.iterrows():
        term = str(row["term"])
        aliases = [a.strip() for a in str(row.get("aliases", "")).split(",") if a.strip()]
        definition = str(row["simple_definition"])
        clean_term = re.sub(r"\s*\([^)]*\)", "", term).strip()
        acronyms = re.findall(r"\(([^)]+)\)", term)

        seed_phrases = [term, clean_term] + acronyms + aliases[:6]
        for phrase in seed_phrases:
            for variant in generate_typo_variants(phrase, rng):
                X_texts.append(variant)
                y_labels.append(idx)

        X_texts.append(normalize_text(f"{term} {clean_term} {definition}"))
        y_labels.append(idx)

    return X_texts, np.array(y_labels, dtype=np.int32)

print("Building augmented training corpus...")
X_texts, y_labels = build_training_corpus(df)
print(f"Generated {len(X_texts):,} training samples across {len(df)} AI terms.")

char_vectorizer = TfidfVectorizer(
    analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True, max_features=8000, dtype=np.float32
)
word_vectorizer = TfidfVectorizer(
    analyzer="word", ngram_range=(1, 3), stop_words="english", sublinear_tf=True, max_features=8000, dtype=np.float32
)

X_char = char_vectorizer.fit_transform(X_texts)

term_reference_docs = []
term_alias_lookup = {}
for idx, row in df.iterrows():
    term = str(row["term"])
    clean_term = re.sub(r"\s*\([^)]*\)", "", term).strip()
    acronyms = re.findall(r"\(([^)]+)\)", term)
    aliases = [a.strip() for a in str(row.get("aliases", "")).split(",") if a.strip()]
    for name in aliases + acronyms + [clean_term, term]:
        norm_name = normalize_text(name)
        comp_name = compact_text(name)
        if norm_name:
            term_alias_lookup[norm_name] = idx
        if comp_name:
            term_alias_lookup[comp_name] = idx
    term_reference_docs.append(normalize_text(f"{term} {clean_term} {' '.join(aliases)} {row['simple_definition']}"))

ref_char_matrix = char_vectorizer.transform([normalize_text(f"{r['term']} {r.get('aliases', '')}") for _, r in df.iterrows()])
ref_word_matrix = word_vectorizer.fit_transform(term_reference_docs)

print("Training Multi-Layer Perceptron (MLPClassifier) Neural Network...")
mlp_model = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation="relu",
    solver="adam",
    alpha=1e-4,
    batch_size=128,
    learning_rate_init=0.003,
    max_iter=25,
    random_state=42,
)
mlp_model.fit(X_char, y_labels)
mlp_model._optimizer = None
mlp_model.coefs_ = [w.astype(np.float32) for w in mlp_model.coefs_]
mlp_model.intercepts_ = [b.astype(np.float32) for b in mlp_model.intercepts_]

train_acc = float(mlp_model.score(X_char, y_labels))
print(f"Training Accuracy: {train_acc * 100:.2f}%")

## 🎯 Step 5: Prediction Function & Student Typo Benchmark Evaluation

In [ ]:
artifact = {
    "df": df,
    "char_vectorizer": char_vectorizer,
    "word_vectorizer": word_vectorizer,
    "mlp_model": mlp_model,
    "ref_char_matrix": ref_char_matrix,
    "ref_word_matrix": ref_word_matrix,
    "term_alias_lookup": term_alias_lookup,
}

def predict_term(query: str, artifact: dict, top_k: int = 3) -> list[dict]:
    df = artifact["df"]
    char_vectorizer = artifact["char_vectorizer"]
    word_vectorizer = artifact["word_vectorizer"]
    mlp_model = artifact["mlp_model"]
    ref_char_matrix = artifact["ref_char_matrix"]
    ref_word_matrix = artifact["ref_word_matrix"]
    term_alias_lookup = artifact["term_alias_lookup"]

    norm_q = normalize_text(query)
    comp_q = compact_text(query)
    if not norm_q:
        return []

    q_char_vec = char_vectorizer.transform([norm_q])
    mlp_probs = mlp_model.predict_proba(q_char_vec)[0]
    full_mlp_probs = np.zeros(len(df), dtype=np.float32)
    full_mlp_probs[mlp_model.classes_] = mlp_probs

    char_sims = cosine_similarity(q_char_vec, ref_char_matrix)[0]
    q_word_vec = word_vectorizer.transform([norm_q])
    word_sims = cosine_similarity(q_word_vec, ref_word_matrix)[0]

    prelim_scores = 0.45 * full_mlp_probs + 0.35 * char_sims + 0.20 * word_sims
    candidate_idxs = np.argsort(prelim_scores)[::-1][:40]

    fuzzy_scores = np.zeros(len(df), dtype=np.float32)
    for idx in candidate_idxs:
        row = df.iloc[int(idx)]
        term_str = str(row["term"])
        clean_term = re.sub(r"\s*\([^)]*\)", "", term_str).strip()
        aliases = [a.strip() for a in str(row.get("aliases", "")).split(",") if a.strip()]
        best_ratio = 0.0
        for cand in [term_str, clean_term] + aliases[:8]:
            r1 = SequenceMatcher(None, norm_q, normalize_text(cand)).ratio()
            r2 = SequenceMatcher(None, comp_q, compact_text(cand)).ratio()
            best_ratio = max(best_ratio, r1, r2)
        fuzzy_scores[int(idx)] = best_ratio

    combined_scores = 0.35 * full_mlp_probs + 0.25 * char_sims + 0.15 * word_sims + 0.25 * fuzzy_scores

    exact_idx = term_alias_lookup.get(norm_q, term_alias_lookup.get(comp_q, None))
    if exact_idx is not None:
        combined_scores[exact_idx] = max(float(combined_scores[exact_idx]), 0.99)

    top_indices = np.argsort(combined_scores)[::-1][:top_k]
    results = []
    for idx in top_indices:
        row = df.iloc[int(idx)]
        results.append({
            "term": str(row["term"]),
            "category": str(row["category"]),
            "simple_definition": str(row["simple_definition"]),
            "analogy": str(row.get("analogy", "")),
            "confidence": float(np.clip(combined_scores[int(idx)], 0.0, 1.0)),
        })
    return results

# Evaluate on student spacing/typo benchmark
benchmark_queries = [
    ("back propagation", "backpropagation"),
    ("backprop", "backpropagation"),
    ("bakcpropagation", "backpropagation"),
    ("gradint descent", "gradient descent"),
    ("sgd", "stochastic gradient descent"),
    ("over fitting", "overfitting"),
    ("under fitting", "underfitting"),
    ("conv net", "convolutional neural network"),
    ("cnn", "convolutional neural network"),
    ("lstm", "long short-term memory"),
    ("attenion mechanism", "attention"),
    ("tranformer", "transformer"),
    ("hallucinations", "hallucination"),
    ("lora", "lora"),
    ("rlhf", "reinforcement learning from human feedback"),
    ("tokenisation", "token"),
    ("word embedding", "embedding"),
    ("random forrest", "random forest"),
    ("k means", "k-means"),
    ("confusion matrx", "confusion matrix"),
    ("f1-score", "f1"),
    ("vanishing gradients", "vanishing gradient"),
]

eval_rows = []
hits = 0
for q, exp in benchmark_queries:
    res = predict_term(q, artifact, top_k=3)[0]
    ok = normalize_text(exp) in normalize_text(res["term"]) or normalize_text(res["term"]) in normalize_text(exp)
    hits += int(ok)
    eval_rows.append({
        "Student Input": q,
        "Matched Term": res["term"],
        "Confidence": f"{res['confidence']*100:.1f}%",
        "Simple Definition": res["simple_definition"][:90] + "...",
        "Match": "✅" if ok else "❌",
    })

eval_df = pd.DataFrame(eval_rows)
print(f"Benchmark Top-1 Accuracy: {hits / len(benchmark_queries) * 100:.1f}% ({hits}/{len(benchmark_queries)})")
with open(MODEL_PATH, "wb") as f:
    pickle.dump(artifact, f)
print(f"Saved trained model to {MODEL_PATH} ({os.path.getsize(MODEL_PATH)/1e6:.2f} MB)")
eval_df

## 🔍 Step 6: Test the Simple Definition Teller Interactively

In [ ]:
def ask_definition_teller(student_input: str):
    results = predict_term(student_input, artifact, top_k=3)
    if not results:
        print("Please enter a valid term.")
        return
    top = results[0]
    print("=" * 75)
    print(f"🔍 Student Input : {student_input}")
    print(f"📘 Matched Term  : {top['term']} ({top['category']}) | Confidence: {top['confidence']*100:.0f}%")
    print("-" * 75)
    print(f"📖 Simple Definition:\n   {top['simple_definition']}")
    if top.get("analogy"):
        print(f"\n💡 Context / Intuition:\n   {top['analogy']}")
    print("=" * 75 + "\n")

# Test with several student queries (including spacing and typos)
for sample_query in ["back propagation", "gradint descent", "over fitting", "transformer", "confusion matrx"]:
    ask_definition_teller(sample_query)

In [ ]:
# Try your own term in Colab!
user_term = input("Type any AI / ML term (e.g., back propagation, CNN, overfitting): ")
if user_term.strip():
    ask_definition_teller(user_term)